In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

### Load Dataset

In [2]:
path = r"/Users/Aisyah Eka R/OneDrive/Documents/TA_10123048/Plan B"
df = pd.read_csv(os.path.join(path, "ibm_watson_categorical_binned.csv"))

In [3]:
# Buat variabel gabungan (Opsi A: product_portfolio = policy_type + coverage)
# Kategori 'Special Auto' disatukan agar tidak terpecah menjadi sel-sel kecil (< 2%)
df["product_portfolio"] = np.where(
    df["policy_type"] == "Special Auto",
    "Special Auto (All)",
    df["policy_type"].str.replace(" Auto", "", regex=False) + " - " + df["coverage"]
)

In [4]:
# Kelompokkan variabel sesuai 4 pilar tujuan penelitian
pillars = {
    "Produk & Kepemilikan": [
        "policy_type", "coverage", "no_of_policies_cat",
        "monthly_premium_cat", "vehicle_class", "vehicle_size"
    ],
    "Profil Risiko & Nilai": [
        "total_claim_cat", "last_claim_cat", "complaints_cat", "clv_cat"
    ],
    "Saluran & Respons": [
        "sales_channel", "renew_offer_type", "response"
    ],
    "Profil Nasabah (Demografi)": [
        "income_cat", "education", "employment_status",
        "marital_status", "gender", "region", "state", "tenure_cat"
    ]
}

In [7]:
# Kandidat variabel kolom acuan (J) yang akan dibandingkan
candidate_targets = ["coverage", "policy_type", "product_portfolio", "no_of_policies_cat"]

summary_rows = []
detailed_rows = []

for target in candidate_targets:
    # Hitung metrik keseimbangan massa marginal kolom (c_j)
    col_props = df[target].value_counts(normalize=True)
    J = len(col_props)
    min_cj = col_props.min() * 100
    norm_entropy = -np.sum(col_props * np.log(col_props)) / np.log(J)
    
    # Tentukan variabel baris yang valid (kecualikan diri sendiri & komponen deterministiknya)
    exclude = {target}
    if target == "product_portfolio":
        exclude.update({"policy_type", "coverage"})
        
    p_vals, v_vals, inertias, in2d_vals, cochran_ok = [], [], [], [], []
    sig_per_pillar = {p_name: [0, 0] for p_name in pillars}  # [sig_count, total_count]
    
    for p_name, vars_list in pillars.items():
        for row_var in vars_list:
            if row_var in exclude:
                continue
                
            ct = pd.crosstab(df[row_var], df[target])
            N_mat = ct.values.astype(float)
            n = N_mat.sum()
            I = N_mat.shape[0]
            
            chi2_stat, p_val, dof, expected = chi2_contingency(ct)
            K = min(I - 1, J - 1)
            total_inertia = chi2_stat / n
            cramers_v = np.sqrt(total_inertia / K)
            pct_e5 = (expected >= 5).mean() * 100
            
            # Hitung proporsi inersia 2D via SVD residual terstandarisasi S
            P = N_mat / n
            r = P.sum(axis=1)
            c = P.sum(axis=0)
            S = np.diag(1.0 / np.sqrt(r)) @ (P - np.outer(r, c)) @ np.diag(1.0 / np.sqrt(c))
            s_vals = np.linalg.svd(S, compute_uv=False)
            eigvals = s_vals[:K] ** 2
            in2d = (eigvals[:min(2, K)].sum() / eigvals.sum() * 100) if eigvals.sum() > 0 else 0.0
            
            is_sig = p_val < 0.05
            sig_per_pillar[p_name][1] += 1
            if is_sig:
                sig_per_pillar[p_name][0] += 1
                
            p_vals.append(p_val)
            v_vals.append(cramers_v)
            inertias.append(total_inertia)
            in2d_vals.append(in2d)
            cochran_ok.append(pct_e5 >= 80.0 and expected.min() >= 1.0)
            
            detailed_rows.append({
                "Kandidat Kolom (J)": target,
                "Pilar": p_name,
                "Variabel Baris": row_var,
                "Dimensi": f"{I}x{J}",
                "Chi-Square": round(chi2_stat, 2),
                "p-value": p_val,
                "Cramer's V": round(cramers_v, 4),
                "Total Inersia": round(total_inertia, 4),
                "Inersia 2D (%)": round(in2d, 1),
                "Signifikan": "Ya" if is_sig else "Tidak"
            })
            
    sig_ratio = np.mean([p < 0.05 for p in p_vals]) * 100
    pillar_summary = " | ".join([f"{k.split()[0]}: {v[0]}/{v[1]}" for k, v in sig_per_pillar.items()])
    
    summary_rows.append({
        "Kandidat Kolom (J)": target,
        "J": J,
        "Min c_j (%)": round(min_cj, 2),
        "Entropi c_j": round(norm_entropy, 3),
        "Lolos Cochran (%)": round(np.mean(cochran_ok) * 100, 1),
        "Pasangan Sig. (%)": round(sig_ratio, 1),
        "Mean Cramer's V": round(np.mean(v_vals), 4),
        "Mean Inersia": round(np.mean(inertias), 4),
        "Mean Inersia 2D (%)": round(np.mean(in2d_vals), 1),
        "Cakupan Sig. per Pilar": pillar_summary
    })

df_summary = pd.DataFrame(summary_rows)
df_detailed = pd.DataFrame(detailed_rows)

print("PERBANDINGAN KUANTITATIF KANDIDAT VARIABEL ACUAN KOLOM (J) UNTUK SCA -> MFACT")
display(df_summary)

PERBANDINGAN KUANTITATIF KANDIDAT VARIABEL ACUAN KOLOM (J) UNTUK SCA -> MFACT


,Kandidat Kolom (J),J,Min c_j (%),Entropi c_j,Lolos Cochran (%),Pasangan Sig. (%),Mean Cramer's V,Mean Inersia,Mean Inersia 2D (%),Cakupan Sig. per Pilar
0,coverage,3,9.02,0.801,100.0,55.0,0.0651,0.0313,100.0,Produk: 2/5 | Profil: 2/4 | Saluran: 2/3 | Pro...
1,policy_type,3,4.14,0.622,100.0,5.0,0.0152,0.0005,100.0,Produk: 0/5 | Profil: 0/4 | Saluran: 0/3 | Pro...
2,product_portfolio,7,1.98,0.785,100.0,36.8,0.0636,0.0331,96.0,Produk: 1/4 | Profil: 2/4 | Saluran: 1/3 | Pro...
3,no_of_policies_cat,3,25.11,0.985,100.0,20.0,0.0590,0.0509,100.0,Produk: 1/5 | Profil: 1/4 | Saluran: 2/3 | Pro...


In [9]:
# Tampilkan rincian variabel baris yang SIGNIFIKAN (p < 0.05) untuk kandidat terbaik
best_target = df_summary.sort_values(
    by=["Pasangan Sig. (%)", "Mean Cramer's V"], ascending=False
).iloc[0]["Kandidat Kolom (J)"]

print(f"SELEKSI VARIABEL BARIS SIGNIFIKAN (p < 0.05) UNTUK KANDIDAT TERBAIK: '{best_target}'")
df_best_sig = df_detailed[
    (df_detailed["Kandidat Kolom (J)"] == best_target) & (df_detailed["p-value"] < 0.05)
].sort_values(by=["Pilar", "Cramer's V"], ascending=[True, False])
display(df_best_sig)

SELEKSI VARIABEL BARIS SIGNIFIKAN (p < 0.05) UNTUK KANDIDAT TERBAIK: 'coverage'


,Kandidat Kolom (J),Pilar,Variabel Baris,Dimensi,Chi-Square,p-value,Cramer's V,Total Inersia,Inersia 2D (%),Signifikan
2,coverage,Produk & Kepemilikan,monthly_premium_cat,4x3,3472.26,0.000000e+00,0.4360,0.3801,100.0,Ya
3,coverage,Produk & Kepemilikan,vehicle_class,6x3,18.31,4.997787e-02,0.0317,0.0020,100.0,Ya
17,coverage,Profil Nasabah (Demografi),region,3x3,31.27,2.696989e-06,0.0414,0.0034,100.0,Ya
14,coverage,Profil Nasabah (Demografi),employment_status,5x3,21.77,5.367160e-03,0.0345,0.0024,100.0,Ya
19,coverage,Profil Nasabah (Demografi),tenure_cat,4x3,20.99,1.845841e-03,0.0339,0.0023,100.0,Ya
13,coverage,Profil Nasabah (Demografi),education,5x3,18.65,1.684090e-02,0.0320,0.0020,100.0,Ya
12,coverage,Profil Nasabah (Demografi),income_cat,4x3,17.41,7.880532e-03,0.0309,0.0019,100.0,Ya
5,coverage,Profil Risiko & Nilai,total_claim_cat,4x3,1213.14,6.872275e-259,0.2577,0.1328,100.0,Ya
8,coverage,Profil Risiko & Nilai,clv_cat,4x3,818.65,1.435715e-173,0.2117,0.0896,100.0,Ya
10,coverage,Saluran & Respons,renew_offer_type,4x3,43.67,8.579306e-08,0.0489,0.0048,100.0,Ya
